# Data types

## Character

| Datatype | Description | Example |
| - | - | - |
| `CHAR(30)` | Fixed-length, blank-padded strings. <br> The string has to be EXACTLY the specified length, in this case, 30 characters - no more, no less. These are right-padded with spaces (to fill up the remaining characters not used by definition of a variable) and always consume the same number of bytes.| State abbreviations - all strings stored in the column are of the same length. |
| `VARCHAR(30)` | Variable-length string. The string can have a length up to the specified limit, such as 10, 20, 25 characters, but no more than e.g. 30 characters. | Varchar is appropriate for free-form data entry, e.g. notes column to hold data about customer interactions with your company's customer service department.  |
| PostgreSQL `text` ; MySQL `tinytext`, `text`, `mediumtext`, `longtext` | To store longer strings such as emails, XML documents. | Note: in MySQL, `tinytext` and `text` aren't normally used. Instead, you can see more often the use of mediumtext and longtext that can be used for storing documents. |

> Note 1: 
> for character data types, use single quotes, not doublequotes. 
> If you need to use a single apostrophe as part of the string, use it two times to escape, e.g. to write a string `O'Brien` you can escape like this: `'O''Brien'`
>
> HOWEVER, in BigQuery, you can use single quotes `'` or double quotes `"`, to represent character data types and datetime. 

> Note 2:
> CHAR and VARCHAR are for storing relatively short text strings. For longer, use text data types

Define character set:
```sql
-- for a variable
VARCHAR(20) CHARACTER SET latin1
-- for the entire database
CREATE DATABASE database1 CHARACTER SET latin1;
```

## Numeric

| Datatype | Description | Example |
| - | - | - |
| `SERIAL` | Auto-increments a number upon inserting a new row. The SERIAL type will make your column an INT with a NOT NULL constraint, and automatically increment the integer when a new row is added. `BIGSERIAL` is the same but has a higher range of possible values.  |
| `BOOLEAN` | `TRUE`, `FALSE` | A column indicating whether a customer order has been shipped |
| `INT` | Whole number. MySQL also has `tinyint`, `smallint`, `mediumint`, `int`, `bigint` | |
| `FLOAT` | Can be `FLOAT(p,s)`, where p is the total number of digits and s is number of allowable digits to the right of the decimal point. For MySQL, can be `FLOAT`, `FLOAT(p,s)`, and for even larger numbers `DOUBLE(p,s)`. | E.g. `FLOAT(4,2)` - handles 17.87, 8.19, but rounds 17.8675 to 17.87 and errors at attempt of storing 178.375 |

> Note 1: 
> The numeric data types can be defined as `unsigned`, meaning that they are greater than or equal to zero.

Some useful functions for doing math in SQL:
```sql
-- Power of a number
-- Works in PostgreSQL, MySQL, BigQuery
POW(base, exponent)
POWER(2, 3) -- 2^3
```

```sql
-- | `POW(2, 3)` | Power; in this example, 2^3. Works in PostgreSQL, MySQL. |
-- | `MOD(<number_to_round/column>, <number-by-which-to-divide>)` | Modulo: check the remainder of the division. In this case, remainder is zero if the number is even. E.g. `MOD(3, 2)`, `MOD(column1, 2)`. Works in MySQL and PostgreSQL. |
-- | `exp(x)` | Calculate the e^x |
-- | `ln(x)` | Calculate the natural log of x |
-- | `sqrt(x)` | Calculate the square root of x |

-- In a column 'comparison' with binary values (0 and 1), calculate percentage that all ones make from the total amount
SELECT ROUND( (SUM(comparison)::numeric / COUNT(comparison)::numeric) * 100 , 2 ) AS immediate_percentage

-- SIGN
-- Show the sign of the signed number
-- `1` if positive, `-1` if negative, and `0` if the number is a zero.
SELECT
  balance,
  SIGN(balance) AS sign
FROM account
-- | balance | sign |
-- | - | - |
-- | 102.21 | 1 |
-- | 0 | 0 |
-- | -122 | -1 |

-- ABS
-- Shows the absolute value of a signed number
SELECT ABS(balance) FROM account
```

## Temporal

In [ ]:
-- set time zone
SET time_zone = 'Europe/Zurich'
-- or
ALTER SESSION TIMEZONE = 'Europe/Zurich'

| Datatype | Description | Example |
| - | - | - |
| `TIMESTAMP` | Used by MySQL, PostgreSQL. Date format: `YYYY-MM-DD HH:MM:SS.MSS`. The TIMESTAMP data type has a range of '1970-01-01 00:00:01' UTC to '2038-01-09 03:14:07' UTC. It has varying properties, depending on the MySQL version and the SQL mode the server is running in. | A column that tracks when a user last modified a particular row in a table. |
| `DATETIME` | MySQL. Date format is like TIMESTAMP.  The DATETIME type is used when you need values that contain both date and time information. MySQL retrieves and displays DATETIME values in 'YYYY-MM-DD HH:MM:SS' format. The supported range is '1000-01-01 00:00:00' to '9999-12-31 23:59:59'. | |
| `DATE` | `YYYY-MM-DD` | Column to hold the expected future shipping date of a customer order. An employee's birth date. |
| `YEAR` | `YYYY` | |
| `TIME` | `HHH:MM:SS` | |

> Date is inserted as string in the format `YYYY-MM-DD`, e.g. `2020-03-23`. MySQL or other servers will automatically convert the string into a date, given that the format of the string matches that of the column in the temporal datatype

To automatically populate a table with the date that a record was inserted:
```sql
-- PostgreSQL, MySQL
CREATE TABLE table1 (
	name TEXT, 
--	date TIMESTAMP
	date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

Different ways of writing time:
| Date | Example |
| - | - |
| `YYYY` | 2014 |
| `MM` | 01 to 12 |
| `DD` | 01 to 31 |
| `HH` | Hour: 00 to 23 |
| `HHH` | Hours (elapsed): -838 to 838 |
| `MI` | Minute: 00 to 59 |
| `SS` | Seconds: 00 to 59 |

**Time zones**

```sql
-- Give current UTC time
-- MySQL
SELECT utc_timestamp();

-- Check time zone settings - global time zone and session time zone
-- SYSTEM = server is using the time zone setting from the server on which the database resides
-- MySQL
SELECT @@global.time_zone, @@session.time_zone
-- Set time zone
SET time_zone = 'Europe/Zurich';
-- PostgreSQL
SHOW timezone;
SELECT current_setting('TIMEZONE');
```

**General**

```sql
-- Get a quarter number for a timestamp
SELECT quarter(datetime)

-- Update a date for a row
UPDATE rental
SET return_date = '2019-09-17' -- or '2019-09-17 15:30:00'
WHERE rental_id = 99999;

-- Gives YYYY-MM-DD HH:MM:SS.MSMS
SELECT NOW();
-- Get years of a person from his birthday
SELECT AGE(NOW(), date_of_birth);
-- Get current date; returns 2022-03-16
-- MySQL, CURDATE(); PostgreSQL, doesn't exist
SELECT CURDATE()
-- PostgreSQL, CURRENT_DATE;
-- MySQL, CURRENT_DATE or CURRENT_DATE()
select CURRENT_DATE() -- returns '2024-11-21'
-- Get current time; format `11:34:22`
SELECT CURTIME()
SELECT CURRENT_TIME()
-- Get current timestamp
SELECT CURRENT_TIMESTAMP
-- returns current date formatted as UNIX time
select UNIX_TIMESTAMP()
```

**DATE_SUB**

Subtract specified interval from the passed date argument.

```sql
-- Subtract 10 days from a date
SELECT DATE_SUB('2017-06-15', INTERVAL 10 DAY);
```

**FORMAT_DATE**

Format date in a specified format:
```sql
WITH temp0 AS (
  select '2025-01-01' rundate 
  union all select '2025-02-01' rundate
),

temp1 AS (
  select CAST(rundate AS DATE) rundate 
  from temp0
)

select
  rundate, 
  format_date('Q%Q', rundate) AS quarter, 
  format_date('%Y', rundate) AS year
FROM temp1
-- [{
--   "rundate": "2025-01-01",
--   "quarter": "Q1",
--   "year": "2025"
-- }, {
--   "rundate": "2025-02-01",
--   "quarter": "Q1",
--   "year": "2025"
-- }]
```



**Other**
```sql
-- Typecasting
-- YYYY-MM-DD
SELECT NOW()::DATE
-- HH:MM:SS.MSMS
SELECT NOW()::TIME
-- An example
(NOW()::DATE + INTERVAL '10 MONTHS')::DATE

-- STR_TO_DATE()
-- Change string format to specified datetime format
-- MySQL
-- Returns DATETIME, DATE, or TIME value depending on the contents of the format string
SELECT STR_TO_DATE('September 17, 2019', '%M %d, %Y'); -- generates DATE format: `YYYY-MM-DD`
```

**DAYNAME()**
- MySQL
- Extracts an entity from a date
- *Instead of this, it is better to use EXTRACT*
```sql
SELECT DAYNAME('2019-09-18') -- > Wednesday
```



```sql
-- Select a part of a date
-- year, month, day, hour, minute, second
SELECT date_part('year', (SELECT date_column_name))

-- Select year and month
SELECT TO_CHAR(order_date, 'YYYY-MM')




-- MySQL
-- if you have date containing minutes, hours, etc. apart from the date itself, you can filter only based on year,month,day like this:
WHERE return_date = date('2005-07-05') 
```

MySQL string formats to date type, e.g. `str_to_date('DEC-21-1980', '%b-%d-%Y')`:
| Formatter | Definition | Example |
| - | - | - |
| `%Y` | The four-digit year | |
| `%y` | The two-digit year | |
| `%M` | The full month name | (January..December) |
| `%m` | The numeric month | (0..12) |
| `%b` | The short month name | Jan, Feb, ... |
| `%W` | The full weekday name | (Sunday..Saturday) |
| `%w` | The numeric day of the week | (0=Sunday..6=Saturday) |
| `%a` | The short weekday name | Sun, Mon, ... |
| `%d` | The numeric day of the month | (01..31) |
| `%j` | The day of year | (001..366) |
| `%H` | The hour of the day, in 24-hour format | (00..23) |
| `%h` | The hour of the day, in 12-hour format | (01..12) |
| `%i` | The minutes within the hour | (00..59) |
| `%s` | The number of seconds | (00.59) |
| `%f` | The number of microseconds | (000000..999999) |
| `%p` | AM or PM | |

Examples:
```sql
-- return records where date equals to specified date
select * from personal_data where birthday = '1977-05-04'
select * from personal_data where birthday = '1977-05-04'::date;

-- Thus, we can order birthdays based only on month and date
-- For example, table like this:
--  id | name         |    date    |
-- ----+--------------+------------+
--  21 | Person 1     | 1971-11-21 |
--  23 | Person 2     | 1989-12-29 |

SELECT * 
FROM notable_dates 
ORDER BY 
  EXTRACT(MONTH FROM date) DESC, 
  EXTRACT(DAY FROM date) DESC;
```

Show total monthly payments for film rentals for 2005 for each quarter and month;
```sql
SELECT 
	QUARTER(payment_date) AS quarter,
	MONTHNAME(payment_date) AS month_nm,
	SUM(amount) AS monthly_sales,
	MAX(SUM(amount)) OVER () AS max_overall_sales,
	MAX(SUM(amount)) OVER (PARTITION BY QUARTER(payment_date)) AS max_qrtr_sales
FROM payment
WHERE YEAR(payment_date) = 2005
GROUP BY 
	quarter(payment_date),
	monthname(payment_date)
;

-- quarter|month_nm|monthly_sales|max_overall_sales|max_qrtr_sales|
-- -------+--------+-------------+-----------------+--------------+
--       2|May     |      4823.44|         28368.91|       9629.89|
--       2|June    |      9629.89|         28368.91|       9629.89|
--       3|July    |     28368.91|         28368.91|      28368.91|
--       3|August  |     24070.14|         28368.91|      28368.91|
```

## Type casting

You can cast datatypes in the ways below:
```sql
-- data types: date, numeric, int, float
-- does NOT work in BigQuery or MySQL,
-- but works in PostgreSQL
SELECT 
  whatever::date, 
  whatever2::numeric
-- or
round( SUM(rating::dec / position::dec)::dec / COUNT(rating)::dec, 2) AS quality

-- or
DATE('2025-01-05')

-- another way
SELECT 
  CAST(sss2.id AS STRING),
  CAST(age AS varchar),
  CAST('123' AS SIGNED INTEGER)
  CAST('2025-01-05' AS date)
```

Complex data types:
```sql
-- List - usually used within a WHERE _ IN <list> clause
('Value1', 'Value2', 'Value3')
```

## NULL

Different meanings / contexts of NULL:
- Not applicable: employee ID column for a transaction that took place at an ATM machine
- Value not yet known: federal ID is not known at the time a customer row is created
- Value undefined: when an account is created for a product that has not yet been added to the database

Rules:
- An expression can *be* null, but it can never *equal* null: 
  - ❌ `WHERE return_date = NULL` does not return any rows
  - ✅ `WHERE return_date IS NULL` or `IS NOT NULL`
- Two nulls are never equal to each other


## Array

BigQuery has two array indexing styles:
- `OFFSET(n)` - zero-based
- `ORDINAL(n)` - one-based.

Examples:

```sql
array[ORDINAL(2)] -- take the element at index 1
array[OFFSET(2)] -- take the element at index 2
```


## Array > column etc.

Convert sql array into rows of a column
```sql
SELECT *
FROM UNNEST(ARRAY[1, 2, 3,
                  4, 5, 6
]) AS id1
-- id1   |
-- ------+
--      1|
--      2|
--      3|
--      4|
--      5|
--      6|
```

Separate string into list items
```sql
-- Separate string into list items
SELECT string_to_array('1 2 3 4', ' ') -- gives you output of one cell like this: {1,2,3,4}
-- Separate and put as values of a column
SELECT unnest(string_to_array('1 2 3 4', ' '))
-- example usage: you have a string containing items you want to look for
select *
from employee
where emp_id in (
	select unnest(string_to_array('100 101 102', ' '))::numeric
)
-- make a database selection as a name
```


# Functions

## Character

### LENGTH

> Works for MySQL and PostgreSQL
>
> Works only with string / character data types

For a specified column, returns the string length of each row

```sql
SELECT LENGTH(name)
FROM person;
-- returns:
-- length| -- int data type
-- ------+
--      9|
--      2|
--      5|
--      5|
```

In [ ]:
WITH temp1 AS (
	SELECT 'Manhattan' city
	UNION ALL
	SELECT 'Alaska' city
)
SELECT 
	city,
	LENGTH(city)
FROM temp1

-- city     |LENGTH(city)|
-- ---------+------------+
-- Manhattan|           9|
-- Alaska   |           6|

### character sets

**CHR**

> PostgreSQL

Based on ASCII code, return UTF-8 character. Check mapping here: https://www.languagetesting.com/windows-alt-codes

```sql
SELECT 
	CHR(97) || CHR(98) || CHR(99) || CHR(100), 
	CHR(126) || 'Buenos d' || CHR(237) || 'as, te gusta az' || CHR(250) || 'car?'

-- ?column?|?column?                      |
-- --------+------------------------------+
-- abcd    |~Buenos días, te gusta azúcar?|
```

**ASCII**

> Works for MySQL and PostgreSQL

Returns the ASCII code / number for a (UTF-8) character

```sql
SELECT ASCII('ñ') -- for instance, in character set UTF-8 it's a character 195


SELECT ASCII('í'), ASCII('ú')
-- ascii|ascii|
-- -----+-----+
--   237|  250|
```


### POSITION

In [ ]:
-- find the location of a substring within a string
SELECT POSITION('characters' IN ('hello characters'))

-- position|
-- --------+
--        7|


### QUOTE

> MySQL

Places quotes around the entire string and adds escapes to any single quotes / apostrophes within the string, normally to use this string in another program.

```sql
SELECT QUOTE(txt1)
FROM table1;
```



In [ ]:
SELECT QUOTE(person_id)
FROM person

-- output:
-- QUOTE(person_id)|
-- ---------------+
-- '58'           |
-- '92'           |
-- '182'          |
-- '118'          |


### TRIM

Removes spaces or specified characters from both ends of a string.

```sql
SELECT TRIM(name) FROM employees;
```

### UPPER

```sql
SELECT UPPER(name)
-- Capitalise the first letter only
SELECT CONCAT(
  UPPER(SUBSTRING(name,1,1)),
  LOWER(SUBSTRING(name, 2, LENGTH(name) - 1))
) AS name
```

### slicing

**Slicing a string**

> Index slicing a string
> Note that index begins with 1 here.

```sql
WITH temp1 AS (
  SELECT 'hello there' textvalue
)

SELECT SUBSTRING(textvalue, 5, 3)
FROM temp1
-- [{
--   "f0_": "o t"
-- }]
```


- MySQL, PostgreSQL
- For slicing
- Indexes in SQL start with 1

In [ ]:
-- Select substring from index 5 to index 10
SELECT SUBSTRING('yes hello world', 5, 5); -- returns 'hello'

-- 'hello' + '****' + 'world' -> 'hello****world'
SELECT CONCAT(
    SUBSTRING('yes hello world', 5, 5), 
    '****', 
    SUBSTRING('yes hello world', 11, 5) 
)


### REPLACE

Basic form:

`REPLACE(string_expression, string_pattern, string_replacement)`

Examples:
```sql
/* Simply substitute whitespaces with underscores
*/
WITH temp1 AS (
	SELECT 'John Wayne' AS full_name
	UNION ALL
	SELECT 'Mayor C Payne' AS full_name
)
SELECT REPLACE(full_name, ' ', '_')
FROM temp1
-- replace      |
-- -------------+
-- John_Wayne   |
-- Mayor_C_Payne|


/* In numbers, replace zeros with nothing
*/

WITH temp1 AS (
    SELECT 2010 AS salary
    UNION ALL
    SELECT 20 AS salary
    UNION ALL
    SELECT 215 AS salary
    UNION ALL
    SELECT 13 AS salary
)

SELECT 
	CAST(
		REPLACE (CAST(salary AS VARCHAR), '0', '')
	AS INT)
FROM temp1
-- replace|
-- -------+
--       21|
--       2|
--     215|
--      13|
```


### regular expressions

There are two ways of writing regular expressions in SQL:
- `LIKE`: simplified REGEXP; is not as powerful, but typically faster than regular expressions.
- `~` or `REGEXP`: True REGEXP

First, very basic regex functions `LEFT`:


**LEFT**

> Works for PostgreSQL, MySQL

```sql
-- Match last names that begin with 'Q'
WHERE LEFT(last_name, 1) = 'Q' 
-- Match last names that begin with 'Qu'
WHERE LEFT(last_name, 2) = 'Qu'
```


**LIKE**

> Works for MySQL, PostgreSQL

General form:
```sql
SELECT * 
FROM courses 
WHERE course LIKE '_lgorithms';
```

| Wildcard character | Meaning |
| --- | --- |
| `%` | any character, any number of times (including 0) |
| `_` | exactly 1 character |

These are used with the SQL keyword `LIKE`

```sql
-- Find any clients who are an LLC
client_name LIKE '%LLC';
-- Case insensitive
LOWER(client_name) LIKE 'david'
-- Find employees born in october
birth_date LIKE '____-10-%';

-- names starting with 'W'
LIKE 'W%'
-- the second letter is 'e'
LIKE '_e%'
-- values with a space in them
LIKE '% %'
-- Value ends with '.com'
LIKE '%.com';
-- last_name like MATTHEWS, WALTERS, WATTS
LIKE '_A_T%S'
-- negative LIKE
NOT LIKE '_lgorithms';
-- case-insensitive
ILIKE, NOT ILIKE
```

> `LIKE` Works for MySQL, PostgreSQL

The regex statement `LIKE` in the `SELECT` clause will return a boolean mask for whether a column matches that regexp.

```sql
SELECT 
  first_name,
  first_name LIKE 'A%' AS starts_with_a
  -- MySQL:      alternatively you can use: `first_name REGEXP '^A.*' AS starts_with_a`
  -- PostgreSQL: alternatively you can use: `first_name ~ '^A.*' AS starts_with_a`
FROM employee;
-- returns for MySQL
-- first_name|starts_with_a| -- bigint
-- ----------+-------------+
-- David     |0            |
-- Angela    |1            |
-- Kelly     |0            |
-- Stanley   |0            |
-- Andy      |1            |

-- returns for PostgreSQL
-- first_name|starts_with_a| -- bool
-- ----------+-------------+
-- David     |false        |
-- Angela    |true         |
-- Kelly     |false        |
-- Stanley   |false        |
-- Andy      |true         |
```

**REGEXP**

> MySQL: `REGEXP` 
> PostgreSQL: `~`
> BigQuery: `REGEXP_CONTAINS`

```sql
SELECT * FROM table1 WHERE name ~ '^Grandfather.+|.+parents.+'
-- Entries start with a vowel
WHERE CITY ~ '^[AEIOUaeiou].*';
WHERE CITY REGEXP '^[aeiou]';
-- Entries that do NOT start with a vowel
WHERE LOWER(CITY) NOT REGEXP '^[aeiou].+'

-- Ends with a vowel
WHERE CITY REGEXP '.+[aeiouAEIOU]$'

-- Does not start or end with a vowel
LOWER(CITY) NOT REGEXP '^[aeiou].+|.+[aeiou]$'
```

> Note that BigQuery works slightly different:

```sql
WITH temp1 AS (
  SELECT 'section: protein: 10%; fat: 5%; another section' descr1, 1 id 
  union all 
  select 'section: protein: 15%; fat: 7%; fibre: 10%' descr2, 2 id 
)
select *
from temp1
where
  REGEXP_CONTAINS(descr1, r'.+protein.+')

```


Note that regex can be also used in the `SELECT` clause:

```sql
WITH temp1 AS (
	SELECT 'Alyx' name UNION ALL 
	SELECT 'Brady' UNION ALL SELECT 'Carrie' UNION ALL SELECT 'Carry'
)

SELECT 
	name,
	name ~ 'y$'
FROM temp1

-- name  |?column?|
-- ------+--------+
-- Alyx  |false   |
-- Brady |true    |
-- Carrie|false   |
-- Carry |true    |
```



**CONTAINS_SUBSTR**

> Works only for BigQuery

```sql
SELECT *
FROM `database.countries`
WHERE CONTAINS_SUBSTR(country_name, 'rus')
-- will find rows which in column `country_name` have 'Cyprus', 'Belarus', 'Russia'
```

**REGEXP_EXTRACT_ALL**

> Works for BigQuery

Example 1:

```sql
with temp1 AS (
  select 1 id, 'we are having a vegetarian snack that is also bio based as usual' search_string union all 
  select 2, 'suitable for vegetarians yes' union all 
  select 3, 'not suitable for anyone' union all 
  select 4, 'this product is bio-based yes' union all 
  select 5, 'the product is bio, yet based on artificial materials'
)

-- returns nested results if multiple matches
SELECT 
  id,
  REGEXP_EXTRACT_ALL(
    LOWER(search_string),
    'vegetarian|bio.based'
  ) AS found_matches
FROM temp1
-- [{
--   "id": "1",
--   "found_matches": ["vegetarian", "bio based"]
-- }, {
--   "id": "2",
--   "found_matches": ["vegetarian"]
-- }, {
--   "id": "3",
--   "found_matches": []
-- }, {
--   "id": "4",
--   "found_matches": ["bio-based"]
-- }, {
--   "id": "5",
--   "found_matches": []
-- }]
```

Example 2:

```sql
with temp1 AS (
  select 1 id, 'we are having a vegetarian snack that is also bio based as usual' search_string union all 
  select 2, 'suitable for vegetarians yes' union all 
  select 3, 'not suitable for anyone' union all 
  select 4, 'this product is bio-based yes' union all 
  select 5, 'the product is bio, yet based on artificial materials'
),

keyword_match AS (
  SELECT 
    id,
    REGEXP_EXTRACT_ALL(
      LOWER(search_string),
      'vegetarian|bio.based'
    ) AS found_matches
  FROM temp1
)

SELECT DISTINCT
  found_matches_unnested AS keywords,
  id 
FROM keyword_match 
CROSS JOIN UNNEST(found_matches) AS found_matches_unnested
  
-- [{
--   "keywords": "vegetarian",
--   "id": "1"
-- }, {
--   "keywords": "bio based",
--   "id": "1"
-- }, {
--   "keywords": "vegetarian",
--   "id": "2"
-- }, {
--   "keywords": "bio-based",
--   "id": "4"
-- }]
```

Example 3 - like example 2 but without cross join:

```sql
with temp1 AS (
  select 1 id, 'we are having a vegetarian snack that is also bio based as usual' search_string union all 
  select 2, 'suitable for vegetarians yes' union all 
  select 3, 'not suitable for anyone' union all 
  select 4, 'this product is bio-based yes' union all 
  select 5, 'the product is bio, yet based on artificial materials'
),

keyword_matches AS (
  SELECT 
    id,
    REGEXP_EXTRACT_ALL(
      LOWER(search_string),
      'vegetarian|bio.based'
    ) AS found_matches
  FROM temp1
)

-- SELECT DISTINCT
--   found_matches_unnested AS keywords,
--   id 
-- FROM keyword_match 
-- CROSS JOIN UNNEST(found_matches) AS found_matches_unnested

SELECT
  id,
  found_matches_unnested
FROM 
  keyword_matches, 
  UNNEST(found_matches) AS found_matches_unnested

-- [{
--   "id": "1",
--   "found_matches_unnested": "vegetarian"
-- }, {
--   "id": "1",
--   "found_matches_unnested": "bio based"
-- }, {
--   "id": "2",
--   "found_matches_unnested": "vegetarian"
-- }, {
--   "id": "4",
--   "found_matches_unnested": "bio-based"
-- }]
```

Note that this function only returns non-overlapping matches, for example, using this function to extract `ana` from `banana` returns only one substring, not two. Also, if one keyword is a subset of another, then one of them won't work unless you use `\\b`, but even then some edge cases won't work as expected - see the example below:

```sql
with temp1 AS (
  select 'this is a chemical free product' search_string
),
temp2 AS (
    SELECT
        search_string, 
        regexp_extract_all(
            LOWER(search_string),
            "\\bfree\\b|\\bchemical.free\\b"
            ) AS p_arr
    FROM temp1
)
select * 
from temp2

-- notice how it only matched "chemical free" and not just the keyword "free";
-- ideally, you would want both to be matched.

-- [{
--   "search_string": "this is a chemical free product",
--   "p_arr": ["chemical free"]
-- }]
```


### concatenation

There exist different way to concatenate in SQL:
- operator `||`
- function `CONCAT`:
  - Can handle any expression that returns a string;
  - In MySQL, will convert numbers, dates, etc. to string format


In [ ]:
SELECT 
	'one ' || 'sheep ' || 'slept.', 
	CONCAT('one ', 'sheep ', 'slept.')

-- ?column?        |concat          |
-- ----------------+----------------+
-- one sheep slept.|one sheep slept.|

In [ ]:
WITH temp1 AS (
	SELECT 1 id, 'John' name UNION ALL SELECT 2 id, 'Jane'
)
SELECT 
	temp1.name || temp1.id || ' : is a name.',
	CONCAT(temp1.name, temp1.id, ' : is a name')
FROM temp1

-- ?column?          |concat           |
-- ------------------+-----------------+
-- John1 : is a name.|John1 : is a name|
-- Jane2 : is a name.|Jane2 : is a name|

In [ ]:
-- Update a table's column by concatenating a string at the end
UPDATE table1
SET col1 = CONCAT(col1, ' additional string')

## Numeric

### ABS & SIGN

In [ ]:
WITH temp1 AS (
	SELECT 10 balance UNION ALL 
	SELECT 0 UNION ALL 
	SELECT -10
)

SELECT 
	balance,
	ABS(balance),
	SIGN(balance)
FROM temp1

-- balance|abs|sign|
-- -------+---+----+
--      10| 10| 1.0|
--       0|  0| 0.0|
--     -10| 10|-1.0|

### MOD

Calculates the remainder when one number is divided into another number.

In [ ]:
SELECT MOD(10, 4) -- -> 2
SELECT 10 % 4 -- -> 2

### POW

In [ ]:
SELECT POW(2, 8) -- -> 2**8 = 256

### Rounding

In [ ]:
WITH temp1 AS (
	SELECT 1 number UNION ALL 
	SELECT 1.1 UNION ALL
	SELECT 1.11 UNION ALL 
	SELECT 1.111 UNION ALL 
	SELECT 1.9 UNION ALL 
	SELECT 1.99 UNION ALL 
	SELECT 1.999 UNION ALL 
	SELECT 9 UNION ALL 
	SELECT 11 UNION ALL 
	SELECT 17 UNION ALL
	SELECT 99 UNION ALL
	SELECT 109
)

SELECT 
	ROUND(number, -2) AS round_minus_2,
	ROUND(number, -1) AS round_minus_1,
	ROUND(number)     AS round_0,
	ROUND(number, 1)  AS round_1,
	FLOOR(number),
	CEIL(number),
	TRUNC(number, -2) AS trunc_minus_2,
	TRUNC(number, -1) AS trunc_minus_1,
	TRUNC(number)     AS trunc_0, -- is called TRUNCATE in some SQL systems 
	TRUNC(number, 1)  AS trunc_1
FROM temp1

-- round_minus_2|round_minus_1|round_0|round_1|floor|ceil|trunc_minus_2|trunc_minus_1|trunc_0|trunc_1|
-- -------------+-------------+-------+-------+-----+----+-------------+-------------+-------+-------+
--             0|            0|      1|    1.0|    1|   1|            0|            0|      1|    1.0|
--             0|            0|      1|    1.1|    1|   2|            0|            0|      1|    1.1|
--             0|            0|      1|    1.1|    1|   2|            0|            0|      1|    1.1|
--             0|            0|      1|    1.1|    1|   2|            0|            0|      1|    1.1|
--             0|            0|      2|    1.9|    1|   2|            0|            0|      1|    1.9|
--             0|            0|      2|    2.0|    1|   2|            0|            0|      1|    1.9|
--             0|            0|      2|    2.0|    1|   2|            0|            0|      1|    1.9|
--             0|           10|      9|    9.0|    9|   9|            0|            0|      9|    9.0|
--             0|           10|     11|   11.0|   11|  11|            0|           10|     11|   11.0|
--             0|           20|     17|   17.0|   17|  17|            0|           10|     17|   17.0|
--           100|          100|     99|   99.0|   99|  99|            0|           90|     99|   99.0|
--           100|          110|    109|  109.0|  109| 109|          100|          100|    109|  109.0|

## Temporal

In [ ]:
-- select current date and time at current location
select current_timestamp;

-- select current date and time at utc
SELECT current_timestamp AT time ZONE 'utc';

-- timezone               |
-- -----------------------+
-- 2026-03-26 21:03:05.458|

### Type casting

In [ ]:
-- CAST
-- is more strict
WITH temp1 AS (
	SELECT '2019-09-17 15:30:00' AS rundate
)

SELECT
	CAST(rundate AS timestamp),
	CAST(rundate AS date),
	CAST(rundate AS time)
FROM temp1

-- rundate                |rundate   |rundate |
-- -----------------------+----------+--------+
-- 2019-09-17 15:30:00.000|2019-09-17|15:30:00|

In [ ]:
-- TO_DATE (PostgreSQL)
-- str_to_date(MySQL)
SELECT to_date('September 17, 2019', 'Month DD, YYYY')

-- to_date   |
-- ----------+
-- 2019-09-17|

### INTERVAL

**INTERVAL**
- Is used to add time period to dates
- Used by PostgreSQL, MySQL
- Interval types (note: in PostgreSQL, can write as singular or plural - DAY or DAYS; in MySQL, only as singular - DAY): `second`, `minute`, `hour`, `day`, `month`, `year`, `minute_second`, `hour_second`, `year_month`


In [ ]:
-- Current date + 5 days

-- PostgreSQL 
SELECT CURRENT_DATE + INTERVAL '5 DAY'

-- MySQL
SELECT CURRENT_DATE() + INTERVAL 5 DAY;
SELECT DATE_ADD(CURRENT_DATE(), INTERVAL 5 DAY);


In [ ]:
-- Time a year ago: 
NOW() - INTERVAL '1 YEAR'; -- PostgreSQL

-- To current date add 1 year and 1 month
current_date + INTERVAL '1 year 1 month'

-- Select birthdays between 1977-05-04 and 30 days before that
SELECT * FROM personal_data 
WHERE birthday < '1977-05-04'::date 
AND birthday > '1977-05-04'::date - INTERVAL '30 DAYS';

In [ ]:
-- Add 3 hours, 27 minutes, and 11 seconds to the current timestamp

-- MySQL
SELECT DATE_ADD(CURRENT_TIMESTAMP(), INTERVAL '3:27:11' HOUR_SECOND)

-- PostgreSQL
SELECT CURRENT_TIMESTAMP + INTERVAL '3:27:11' HOUR_SECOND;

-- MySQL: add 9 years and 11 months to a birth_date
UPDATE employee
SET birth_date = DATE_ADD(birth_date, INTERVAL '9-11' YEAR_MONTH)
WHERE emp_id = 123;

### LAST_DAY

In [ ]:
-- MySQL, get last day of the month on the specified date
SELECT LAST_DAY('2019-09-17')

In [ ]:
-- PostgreSQL doesn't have an equivalent built-in function,
-- but you can use the following code
WITH temp1 AS (
	SELECT CAST('2019-09-17 15:30:00' AS timestamp) AS rundate
)

SELECT 
    (
        date_trunc('MONTH', rundate) 
        + INTERVAL '1 MONTH - 1 day'
    )::date AS end_of_month
FROM temp1

-- end_of_month|
-- ------------+
--   2019-09-30|

### DAYNAME

In [ ]:
-- MySQL, dayname - extract day from the date
SELECT dayname('2019-09-18')

In [ ]:
-- PostgreSQL
WITH temp1 AS (
	SELECT CAST('2019-09-17 15:30:00' AS timestamp) AS rundate
)

SELECT to_char(rundate, 'Day')
FROM temp1

-- to_char  |
-- ---------+
-- Tuesday  |

### EXTRACT


- Extracts fields: DAY, DOW, MONTH, YEAR, CENTURY
- EXTRACT can be used in SELECT and WHERE
- PostgreSQL, MySQL


In [ ]:
-- EXTRACT: Extracting fields: DAY, DOW, WEEK, MONTH, YEAR, CENTURY
-- EXTRACT can be used in SELECT and WHERE

-- MySQL
EXTRACT(YEAR FROM CURRENT_DATE())
-- MySQL
SELECT YEAR(NOW()), MONTH(NOW()), DAY(NOW()), HOUR(NOW()), MINUTE(NOW()), SECOND(NOW())
-- Select month of February
SELECT * FROM notable_dates WHERE EXTRACT (MONTH FROM date1) = 02
-- Compare two years
WHERE EXTRACT(YEAR FROM date1) < EXTRACT(YEAR FROM CURRENT_DATE())
WHERE EXTRACT(YEAR FROM e.birth_date) IN (1967, 1961)
-- MySQL
YEAR(date1) = 2004 -- or '2004'
QUARTER(date1)
MONTHNAME(date1)

In [ ]:
-- PostgreSQL
WITH temp1 AS (
	SELECT CAST('2019-09-17 15:30:00' AS timestamp) AS rundate
)

SELECT 
	EXTRACT(YEAR FROM NOW()),
	EXTRACT(YEAR FROM rundate)
FROM temp1

-- extract|extract|
-- -------+-------+
--    2026|   2019|

### DATEDIFF

In [ ]:
-- DATEDIFF
-- MySQL
-- returns the number of full days between two dates
-- ignores the time of day in its arguments
SELECT DATEDIFF('2019-09-03', '2019-06-21') -- > 74
SELECT DATEDIFF('2019-09-03 23:59:59', '2019-06-21 00:00:01') -- > 74
SELECT DATEDIFF('2019-06-21', '2019-09-03') -- > -74
-- in PostgreSQL, you just subtract
SELECT '2025-09-20'::date - '2025-09-01'::date AS days_difference; -- -> 19 -- days
-- in BigQuery, slight variation
DATE_DIFF('2019-10-01', '2019-05-01', DAY)

# Window functions

> a.k.a. data windows, window functions, analytic function

In SQL, a window function allows advanced analytics by letting you calculate across rows while keeping all data intact; it is a function which uses values from one or multiple rows to return a value for each row. 
- ℹ️ This contrasts with an aggregate function, which returns a single value for multiple rows.

Window functions have an OVER clause; any function without an OVER clause is not a window function, but rather an aggregate or single-row (scalar) function.

Anatomy of a window function:
- Some aggregate function first;
- `OVER()` clause: defines the set of rows for the function. Every window function requires this;
- `PARTITION BY`: split data into groups, applying the function within each group. Similar to GROUP BY but without collapsing rows;
- `ORDER BY`: orders rows within each partition, critical for functions like ROW_NUMBER() and RANK();

```sql
<function_name>() OVER (
  PARTITION BY <column> 
  ORDER BY <column>
)
```

**Example 0**

``sql
WITH temp1 AS (
  SELECT 100 price, 1 id 
  union all 
  select 150 price, 2 id 
  union all
  select 200 price, 2 id 
)
select
  id, 
  price, 
  SUM(price) OVER (PARTITION BY id) AS sum_price
FROM temp1
<!-- Fila	id	price	sum_price
1	1	100	100
2	2	150	350
3	2	200	350	 -->
```

**Example 1:**

Show total monthly payments for film rentals for 2005 for each quarter and month;
```sql
SELECT 
	QUARTER(payment_date) AS quarter,
	MONTHNAME(payment_date) AS month_nm,
	SUM(amount) AS monthly_sales,
	MAX(SUM(amount)) OVER () AS max_overall_sales,
	MAX(SUM(amount)) OVER (PARTITION BY QUARTER(payment_date)) AS max_qrtr_sales
FROM payment
WHERE YEAR(payment_date) = 2005
GROUP BY 
	quarter(payment_date),
	monthname(payment_date)
;

-- quarter|month_nm|monthly_sales|max_overall_sales|max_qrtr_sales|
-- -------+--------+-------------+-----------------+--------------+
--       2|May     |      4823.44|         28368.91|       9629.89|
--       2|June    |      9629.89|         28368.91|       9629.89|
--       3|July    |     28368.91|         28368.91|      28368.91|
--       3|August  |     24070.14|         28368.91|      28368.91|
```

**Example 2:**

```sql
SELECT 
	year,
	month, 
	MAX(MAX(sale)) OVER () AS max_overall_sale,
	MAX(SUM(sale)) OVER (PARTITION BY month) AS total_monthly_sales,
	MAX(MAX(sale)) OVER (PARTITION BY year, month) AS max_monthly_sales
	
FROM (
	SELECT '2014' AS year, 'June' AS month, 100 AS sale
	UNION ALL
	SELECT '2014' AS year, 'June' AS month, 150 AS sale
	UNION ALL
	SELECT '2014' AS year, 'July' AS month, 170 AS sale
	UNION ALL
	SELECT '2015' AS year, 'June' AS month, 300 AS sale
	UNION ALL
	SELECT '2015' AS year, 'June' AS month, 50 AS sale
) a1
GROUP BY 
	year,
	month

-- Result: 
-- year|month|max_overall_sale|total_monthly_sales|max_monthly_sales|
-- ----+-----+----------------+-------------------+-----------------+
-- 2014|July |             300|                170|              170|
-- 2014|June |             300|                350|              150|
-- 2015|June |             300|                350|              300|
```


#### RANK

Ranking functions:
- `ROW_NUMBER`: returns a unique number for each row
- `RANK`: returns the same ranking in case of a tie, with gaps in the ranking
- `DENSE_RANK`: returns the same ranking in case of a tie, with NO gaps in the ranking

> For many situations, the `RANK` function might be the best option

**A clear comparative example**

```sql
SELECT 
	customer_id,
	num_rentals,
	ROW_NUMBER() OVER (ORDER BY num_rentals DESC) AS row_number_rnk,
	RANK() OVER (ORDER BY num_rentals DESC) AS rank_rnk,
	DENSE_RANK() OVER (ORDER BY num_rentals DESC) AS dense_rank_rnk
FROM (	
	SELECT 1 customer_id, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 43 num_rentals
) a1;
-- customer_id|num_rentals|row_number_rnk|rank_rnk|dense_rank_rnk|
-- -----------+-----------+--------------+--------+--------------+
--           1|         46|             1|       1|             1|
--           2|         45|             2|       2|             2|
--           3|         45|             3|       2|             2|
--           4|         44|             4|       4|             3|
--           5|         44|             5|       4|             3|
--           6|         44|             6|       4|             3|
--           7|         43|             7|       7|             4|
```


Another example: **For each group, save only the longest string**

```sql
WITH a1 AS (
	SELECT 'Text 1' AS texts, 1 AS groups
	UNION ALL
	SELECT 'Longer texts' AS texts, 1 AS groups
	UNION ALL
	SELECT 'fasdd asdfasd' AS texts, 2 AS groups
	UNION ALL
	SELECT 'aas algitorkdm' AS texts, 2 AS groups
	UNION ALL 
	SELECT NULL AS texts, 3 AS groups
	UNION ALL
	SELECT 'aaa' AS texts, 3 AS groups
),
a2 AS (
	SELECT 
		groups,
		CASE WHEN texts IS NULL THEN '-' ELSE texts END AS texts
	FROM a1
),
a3 AS (
	SELECT
		texts, 
		groups,
		ROW_NUMBER() OVER (PARTITION BY groups ORDER BY LENGTH(texts) DESC)
	FROM a2
)
SELECT *
FROM a3
WHERE row_number = 1
```


Get top 3 for ROW_NUMBER() function:
```sql
SELECT *
FROM (
	SELECT 
		customer_id,
		num_rentals,
		ROW_NUMBER() OVER (ORDER BY num_rentals DESC) AS row_number_rnk
	FROM (	
		SELECT 1 customer_id, 46 num_rentals
		UNION ALL
		SELECT 2 customer_id, 45 num_rentals
		UNION ALL
		SELECT 3 customer_id, 45 num_rentals
		UNION ALL
		SELECT 4 customer_id, 44 num_rentals
		UNION ALL
		SELECT 5 customer_id, 44 num_rentals
		UNION ALL
		SELECT 6 customer_id, 44 num_rentals
		UNION ALL
		SELECT 7 customer_id, 43 num_rentals
	) a1
) a2
WHERE 
	row_number_rnk <= 3
;
-- customer_id|num_rentals|row_number_rnk|
-- -----------+-----------+--------------+
--           1|         46|             1|
--           2|         45|             2|
--           3|         45|             3|
```

Multiple ranking - get ranking for each year
```sql
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT 
	customer_id,
	yr,
	num_rentals,
	ROW_NUMBER() OVER (PARTITION BY yr ORDER BY num_rentals DESC) AS num_rentals_rank
FROM temp1

-- customer_id|yr  |num_rentals|dense_rank_rnk|
-- -----------+----+-----------+--------------+
--           1|2014|         46|             1|
--           2|2014|         45|             2|
--           3|2014|         45|             2|
--           4|2014|         44|             3|
--           5|2015|         44|             1|
--           6|2015|         44|             1|
--           7|2015|         43|             2|
```

#### QUALIFY

QUALIFY is a clause used to filter the results of a window function. 

> You need a QUALIFY statement because WHERE, GROUP BY, and HAVING filtering statements are all evaluated before the window functions. This can be overcome either by QUALIFY within the same query or writing filters on a window function-containing query contained within a CTE.

Examples:
> QUALIFY is available in Snowflake, Databricks, ClickHouse, BigQuery, Redshift
> QUALIFY doesn't work in PostgreSQL or MySQL

**Basic usage example**:

```sql
/* This is the way you filter the lowest num_rentals
per year, using CTEs
*/
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
),

ranked_orders AS (
	SELECT
		*,
		ROW_NUMBER() OVER (
			PARTITION BY yr
			ORDER BY num_rentals ASC 
		) AS rn 
	FROM temp1
)

SELECT *
FROM ranked_orders
WHERE rn = 1;

-- customer_id|yr  |num_rentals|rn|
-- -----------+----+-----------+--+
--           4|2014|         44| 1|
--           7|2015|         43| 1|

/* However, you can do the same as above 
in one step (instead of two) 
using QUALIFY statement
*/
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT *
FROM temp1
QUALIFY ROW_NUMBER() OVER(
	PARTITION BY yr 
	ORDER BY num_rentals ASC
) = 1
```


```sql
-- Notice in this example that QUALIFY is applied AFTER 
-- apllying the WHERE clause
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT *
FROM temp1
WHERE customer_id IN (1, 2, 5, 6)
QUALIFY ROW_NUMBER() OVER(
	PARTITION BY yr 
	ORDER BY num_rentals ASC
) = 1
-- | customer_id | yr   | num_rentals |
-- | ----------- | ---- | ----------- |
-- | 2           | 2014 | 45          |
-- | 5           | 2015 | 44          |

-- if you comment out the WHERE clause, then apply QUALIFY in a separate CTE, and then apply WHERE, 
-- the results will be different - will return empty table
```

#### LAG, LEAD

```sql
WITH temp1 AS (
	SELECT 'Lisa' name, '2021-01-01' date, 5500 salary
	UNION ALL
	SELECT 'Lisa' name, '2022-01-01' date, 7000 salary
	UNION ALL
	SELECT 'Lisa' name, '2023-01-01' date, 7500 salary
	UNION ALL
	SELECT 'Lisa' name, '2024-01-01' date, 8000 salary
)
SELECT
	name, 
	date,
	salary,
	LAG(salary) OVER (ORDER BY date) AS prev_salary,
	LEAD(salary) OVER (ORDER BY date) AS next_salary
FROM temp1;

-- output
-- name|date      |salary|prev_salary|next_salary|
-- ----+----------+------+-----------+-----------+
-- Lisa|2021-01-01|  5500|           |       7000|
-- Lisa|2022-01-01|  7000|       5500|       7500|
-- Lisa|2023-01-01|  7500|       7000|       8000|
-- Lisa|2024-01-01|  8000|       7500|           |
```

#### Cumulative SUM

```sql
WITH daily_sales AS (
	SELECT '2023-05-01'::date date, 100 sales
	UNION ALL
	SELECT '2023-05-02'::date date, 200 sales
	UNION ALL
	SELECT '2023-05-03'::date date, 150 sales
)
SELECT 
	date, 
	sales,
	SUM(sales) OVER (ORDER BY Date) as CumulativeSales
FROM daily_sales;

-- date      |sales|cumulativesales|
-- ----------+-----+---------------+
-- 2023-05-01|  100|            100|
-- 2023-05-02|  200|            300|
-- 2023-05-03|  150|            450|
```

#### FIRST_VALUE

FIRST_VALUE() returns the first value in an ordered partition, while LAST_VALUE() returns the last value. 

𝗙𝗼𝗿 𝗲𝘅𝗮𝗺𝗽𝗹𝗲: In analyzing stock prices, FIRST_VALUE() can be used to compare daily stock prices to the price at month's start, so we can measure price changes relative to the month's opening price.

Example:

**FIRST_VALUE() behaves exactly as expected:**
```sql
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT 
	customer_id,
	yr,
	num_rentals,
	FIRST_VALUE(num_rentals) OVER (PARTITION BY yr ORDER BY num_rentals)
FROM temp1;
-- customer_id|yr  |num_rentals|first_value|
-- -----------+----+-----------+-----------+
--           4|2014|         44|         44|
--           2|2014|         45|         44|
--           3|2014|         45|         44|
--           1|2014|         46|         44|
--           7|2015|         43|         43|
--           5|2015|         44|         43|
--           6|2015|         44|         43|

```

**LAST_VALUE(), however, has an interesting case:**

```sql
/*
If you run it like below, it will find the last row between the start and the current row for each partition
*/
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT 
	customer_id,
	yr,
	num_rentals,
	LAST_VALUE(num_rentals) OVER (PARTITION BY yr ORDER BY num_rentals)
FROM temp1
-- customer_id|yr  |num_rentals|last_value|
-- -----------+----+-----------+----------+
--           4|2014|         44|        44|
--           2|2014|         45|        45|
--           3|2014|         45|        45|
--           1|2014|         46|        46|
--           7|2015|         43|        43|
--           5|2015|         44|        44|
--           6|2015|         44|        44|

/*
This is because by default, if you don't specify the range for the LAST_VALUE function, it runs the following: 
LAST_VALUE(num_rentals) OVER (PARTITION BY yr ORDER BY num_rentals RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
which exactly finds the last value in the range between the starting value and the current row's value.

Nevertheless, if you (as expected) want to find the last row in the range of starting value to the end value of a partition, 
you need to specify it in the range:
*/
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT 
	customer_id,
	yr,
	num_rentals,
	LAST_VALUE(num_rentals) OVER (PARTITION BY yr ORDER BY num_rentals RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
FROM temp1
-- customer_id|yr  |num_rentals|last_value|
-- -----------+----+-----------+----------+
--           4|2014|         44|        46|
--           2|2014|         45|        46|
--           3|2014|         45|        46|
--           1|2014|         46|        46|
--           7|2015|         43|        44|
--           5|2015|         44|        44|
--           6|2015|         44|        44|
```

#### Sliding window

E.g. this leet code question: https://leetcode.com/problems/restaurant-growth/submissions/1649171860/?envType=study-plan-v2&envId=top-sql-50

Query below answer the question: 
- What is the cumulative sum of current day with the previous day for purchases? 
- What is the average purchase between each day and it's previous day? 

```sql
WITH temp1 AS (
  SELECT '2024-01-01' date, 10 purchase
  UNION ALL
  SELECT '2024-01-02' date, 50 purchase
  UNION ALL
  SELECT '2024-01-02' date, 50 purchase
  UNION ALL
  SELECT '2024-01-03' date, 60 purchase
  UNION ALL
  SELECT '2024-01-04' date, 70 purchase
  UNION ALL
  SELECT '2024-01-04' date, 10 purchase
), 
temp2 AS (
  SELECT 
    CAST(date AS date) AS date,
    purchase
  FROM temp1
)

SELECT 
  date,
  purchase,
  SUM(purchase) OVER (
    ORDER BY date 
    RANGE BETWEEN INTERVAL 1 DAY PRECEDING 
    AND CURRENT ROW 
  ) AS cumsum_prev_day,
  ROUND(
    AVG(purchase) OVER (
      ORDER BY date 
      RANGE BETWEEN INTERVAL 1 DAY PRECEDING 
      AND CURRENT ROW)
  , 2) AS cumavg_prev_day
FROM temp2
```